![Cartoon of telecom customers](IMG_8811.png)


The telecommunications (telecom) sector in India is rapidly changing, with more and more telecom businesses being created and many customers deciding to switch between providers. "Churn" refers to the process where customers or subscribers stop using a company's services or products. Understanding the factors that influence keeping a customer as a client in predicting churn is crucial for telecom companies to enhance their service quality and customer satisfaction. As the data scientist on this project, you aim to explore the intricate dynamics of customer behavior and demographics in the Indian telecom sector in predicting customer churn, utilizing two comprehensive datasets from four major telecom partners: Airtel, Reliance Jio, Vodafone, and BSNL:

- `telecom_demographics.csv` contains information related to Indian customer demographics:

| Variable             | Description                                      |
|----------------------|--------------------------------------------------|
| `customer_id `         | Unique identifier for each customer.             |
| `telecom_partner `     | The telecom partner associated with the customer.|
| `gender `              | The gender of the customer.                      |
| `age `                 | The age of the customer.                         |
| `state`                | The Indian state in which the customer is located.|
| `city`                 | The city in which the customer is located.       |
| `pincode`              | The pincode of the customer's location.          |
| `registration_event` | When the customer registered with the telecom partner.|
| `num_dependents`      | The number of dependents (e.g., children) the customer has.|
| `estimated_salary`     | The customer's estimated salary.                 |

- `telecom_usage` contains information about the usage patterns of Indian customers:

| Variable   | Description                                                  |
|------------|--------------------------------------------------------------|
| `customer_id` | Unique identifier for each customer.                         |
| `calls_made` | The number of calls made by the customer.                    |
| `sms_sent`   | The number of SMS messages sent by the customer.             |
| `data_used`  | The amount of data used by the customer.                     |
| `churn`    | Binary variable indicating whether the customer has churned or not (1 = churned, 0 = not churned).|


In [46]:
# Import libraries and methods/functions
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeClassifier,LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Start your code here!

In [60]:
dfd=pd.read_csv('telecom_demographics.csv')
dfu=pd.read_csv('telecom_usage.csv')

      customer_id telecom_partner gender  age             state       city  \
0           15169          Airtel      F   26  Himachal Pradesh      Delhi   
1          149207          Airtel      F   74       Uttarakhand  Hyderabad   
2          148119          Airtel      F   54         Jharkhand    Chennai   
3          187288    Reliance Jio      M   29             Bihar  Hyderabad   
4           14016        Vodafone      M   45          Nagaland  Bangalore   
...           ...             ...    ...  ...               ...        ...   
6495        78836          Airtel      M   54            Odisha    Chennai   
6496       146521            BSNL      M   69    Andhra Pradesh  Hyderabad   
6497        40413          Airtel      M   19           Gujarat  Hyderabad   
6498        64961        Vodafone      M   26         Meghalaya    Chennai   
6499        60427    Reliance Jio      M   19           Mizoram  Hyderabad   

      pincode registration_event  num_dependents  estimated_sal

In [61]:
churn_df=pd.merge(dfd,dfu)
churn_df

,customer_id,telecom_partner,gender,age,state,city,pincode,registration_event,num_dependents,estimated_salary,calls_made,sms_sent,data_used,churn
0,15169,Airtel,F,26,Himachal Pradesh,Delhi,667173,2020-03-16,4,85979,75,21,4532,1
1,149207,Airtel,F,74,Uttarakhand,Hyderabad,313997,2022-01-16,0,69445,35,38,723,1
2,148119,Airtel,F,54,Jharkhand,Chennai,549925,2022-01-11,2,75949,70,47,4688,1
3,187288,Reliance Jio,M,29,Bihar,Hyderabad,230636,2022-07-26,3,34272,95,32,10241,1
4,14016,Vodafone,M,45,Nagaland,Bangalore,188036,2020-03-11,4,34157,66,23,5246,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6495,78836,Airtel,M,54,Odisha,Chennai,125785,2021-01-29,4,124805,-2,39,5000,0
6496,146521,BSNL,M,69,Andhra Pradesh,Hyderabad,923076,2022-01-03,1,65605,20,31,3562,0
6497,40413,Airtel,M,19,Gujarat,Hyderabad,152201,2020-07-21,0,28632,73,14,65,0
6498,64961,Vodafone,M,26,Meghalaya,Chennai,782127,2020-11-21,3,119757,52,8,6835,0


In [63]:
churn_rate=churn_df['churn'].value_counts()/len(churn_df)
churn_rate

churn
0    0.799538
1    0.200462
Name: count, dtype: float64

In [64]:
churn_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6500 entries, 0 to 6499
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   customer_id         6500 non-null   int64 
 1   telecom_partner     6500 non-null   object
 2   gender              6500 non-null   object
 3   age                 6500 non-null   int64 
 4   state               6500 non-null   object
 5   city                6500 non-null   object
 6   pincode             6500 non-null   int64 
 7   registration_event  6500 non-null   object
 8   num_dependents      6500 non-null   int64 
 9   estimated_salary    6500 non-null   int64 
 10  calls_made          6500 non-null   int64 
 11  sms_sent            6500 non-null   int64 
 12  data_used           6500 non-null   int64 
 13  churn               6500 non-null   int64 
dtypes: int64(9), object(5)
memory usage: 711.1+ KB


In [13]:
# churn_df['registration_event']=pd.to_datetime(churn_df['registration_event'])

In [65]:
categorical_col=churn_df.select_dtypes('object').columns

In [66]:
churn_df=pd.get_dummies(churn_df,columns=categorical_col)
features=churn_df.drop(['customer_id','churn'],axis=1)
scaler=StandardScaler()
features_scaled=scaler.fit_transform(features)
target=churn_df['churn']

In [69]:
x_train,x_test,y_train,y_test=train_test_split(features_scaled,target,test_size=0.2,random_state=42)

print(x_train.shape,x_test.shape,y_train.shape,y_test.shape)

(5200, 1263) (1300, 1263) (5200,) (1300,)


In [72]:
logreg=LogisticRegression(random_state=42)

logreg.fit(x_train,y_train)

,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,class_weight,None
,solver,'auto'
,positive,False
,random_state,42


In [73]:
logreg_pred=logreg.predict(x_test)

from sklearn.metrics import accuracy_score
logacc=accuracy_score(y_test,logreg_pred)
print(logacc)


0.7323076923076923


In [76]:
print(confusion_matrix(y_test,logreg_pred))

[[927 100]
 [248  25]]


In [78]:
print(classification_report(y_test,logreg_pred))

              precision    recall  f1-score   support

           0       0.79      0.90      0.84      1027
           1       0.20      0.09      0.13       273

    accuracy                           0.73      1300
   macro avg       0.49      0.50      0.48      1300
weighted avg       0.67      0.73      0.69      1300



In [74]:
rfc=RandomForestClassifier(random_state=42)
rfc.fit(x_train,y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [75]:
rf_pred=rfc.predict(x_test)
accr=accuracy_score(y_test,rf_pred)
accr

0.7892307692307692

In [77]:
print(confusion_matrix(y_test,rf_pred))

[[1026    1]
 [ 273    0]]


In [79]:
print(classification_report(y_test,rf_pred))

              precision    recall  f1-score   support

           0       0.79      1.00      0.88      1027
           1       0.00      0.00      0.00       273

    accuracy                           0.79      1300
   macro avg       0.39      0.50      0.44      1300
weighted avg       0.62      0.79      0.70      1300



In [80]:
higher_accuracy="RandomForest"